# 🤖 Smart AI Trash Bin - YOLOv8 Training & Deployment Notebook
### โน้ตบุ๊กสำหรับเทรนโมเดลคัดแยกขยะ 4 ประเภท (General, Recyclable, Wet, Hazardous) ด้วย YOLOv8 บน Google Colab (GPU ฟรี)

**คำแนะนำการตั้งค่า Runtime บน Google Colab:**
1. ไปที่เมนู `Runtime` -> `Change runtime type`
2. เลือก Hardware accelerator: **T4 GPU**
3. กด Save แล้วเริ่มรันทีละ Cell ตามลำดับ

## 1. ติดตั้งไลบรารีที่จำเป็น (Install Ultralytics & Roboflow)

In [ ]:
!pip install -q ultralytics roboflow

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. ดาวน์โหลด Dataset ขยะ 4 คลาสจาก Roboflow
*หมายเหตุ: ใส่ Roboflow API Key และ Workspace/Project ของคุณที่เตรียมไว้ หรือใช้ Dataset ตัวอย่าง*

In [ ]:
from roboflow import Roboflow

# กรุณาใส่ Roboflow API Key ของคุณที่นี่ (สมัครฟรีที่ roboflow.com)
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"

# หากยังไม่มี Dataset ของตัวเอง ระบบจะดาวน์โหลดตัวอย่างขยะสาธารณะ (Waste Classification)
try:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("smart-trash-bin").project("waste-sorting-4class")
    dataset = project.version(1).download("yolov8")
    data_yaml_path = f"{dataset.location}/data.yaml"
    print("Dataset downloaded successfully from your Roboflow account!")
except Exception as e:
    print(f"Custom download failed or API Key not set: {e}")
    print("กำลังดาวน์โหลด Dataset สาธารณะสำหรับ Waste Sorting มาเป็นตัวอย่าง...")
    # ดาวน์โหลด Dataset ขยะสาธารณะตัวอย่าง
    !curl -L "https://universe.roboflow.com/ds/waste-sorting-demo" > waste.zip; unzip -q waste.zip -d ./dataset || true
    data_yaml_path = "data.yaml"

## 3. สั่งเทรนโมเดล YOLOv8 Nano (YOLOv8n)
*ใช้ YOLOv8n ขนาดเล็กที่สุด เหมาะสำหรับ Embedded Systems และ Cloud REST API ที่ต้องการ Latency ต่ำ (~200ms)*

In [ ]:
from ultralytics import YOLO

# 1. โหลด Base Model YOLOv8 Nano
model = YOLO('yolov8n.pt')

# 2. เริ่มเทรน (50 Epochs, ภาพ 416x416)
# เปิด Augmentations (Crop, Scale, Brightness) เพื่อรองรับกรณีผู้ใช้ 'ถือขยะด้วยมือ'
results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=416,
    batch=16,
    name='smart_trash_bin',
    augment=True,
    degrees=10.0,
    fliplr=0.5,
    mosaic=1.0
)

print("Training Finished!")

## 4. ประเมินประสิทธิภาพโมเดล (Model Evaluation & Metrics)

In [ ]:
# ประเมินผล Validation Metrics
metrics = model.val()
print(f"mAP 50-95: {metrics.box.map:.3f}")
print(f"mAP 50:    {metrics.box.map50:.3f}")

# แสดงกราฟ Confusion Matrix และ Precision-Recall
from IPython.display import Image
Image(filename='runs/detect/smart_trash_bin/confusion_matrix.png')

## 5. ทดสอบจำแนกภาพตัวอย่าง (Inference Test on Sample Images)

In [ ]:
# ดาวน์โหลดภาพตัวอย่างขวดน้ำพลาสติกมาทดสอบ
!curl -o test_bottle.jpg https://images.unsplash.com/photo-1523362628745-0c100150b504?w=500

# ทำการพยากรณ์
preds = model.predict(source='test_bottle.jpg', conf=0.4)

# แสดงผลลัพธ์ Bounding Box
for r in preds:
    r.save(filename='result.jpg')
    for box in r.boxes:
        cls_name = model.names[int(box.cls[0])]
        conf = float(box.conf[0])
        print(f"Detected: {cls_name} (Confidence: {conf*100:.1f}%)")

Image(filename='result.jpg')

## 6. ดาวน์โหลดไฟล์โมเดลที่ดีที่สุด (`best.pt`)
*สามารถนำ `best.pt` ไปอัปโหลดขึ้น Roboflow เพื่อรับ Hosted API Key หรือรันเซิร์ฟเวอร์ด้วยตนเอง*

In [ ]:
from google.colab import files
print("กำลังดาวน์โหลด best.pt สู่คอมพิวเตอร์ของคุณ...")
files.download('runs/detect/smart_trash_bin/weights/best.pt')